# Module 11: Sequential logic
A notebook that checks the module's numbers (Floyd Ch.7 to 10). Each part: a question, code, then a "Reading the result" cell quoting the numbers in the output.

## 1. The S-R latch and the J-K flip-flop
What does the NOR latch do for (S, R) = (1,0), (0,0), (0,1), (0,0), (1,1), and does the J-K table match the characteristic equation?

In [1]:
# Chot S-R NOR (tac dung muc CAO): mo phong cong lap den khi on dinh
def sr_nor(S, R, Q, Qb):
    for _ in range(10):
        q2 = int(not (R or Qb)); qb2 = int(not (S or q2))
        if (q2, qb2) == (Q, Qb): break
        Q, Qb = q2, qb2
    return Q, Qb
Q, Qb = 0, 1
for S_, R_ in [(1, 0), (0, 0), (0, 1), (0, 0), (1, 1)]:
    Q, Qb = sr_nor(S_, R_, Q, Qb); print('S R =', S_, R_, '-> Q, Qbar =', Q, Qb)
# J-K: bang so voi phuong trinh dac trung Q+ = J.Qbar + Kbar.Q
table = {(0, 0): lambda q: q, (0, 1): lambda q: 0, (1, 0): lambda q: 1, (1, 1): lambda q: 1 - q}
ok = all(table[(j, k)](q) == int((j and not q) or ((not k) and q)) for j in (0, 1) for k in (0, 1) for q in (0, 1))
print('bang J-K = phuong trinh dac trung (8 to hop):', ok)

S R = 1 0 -> Q, Qbar = 1 0
S R = 0 0 -> Q, Qbar = 1 0
S R = 0 1 -> Q, Qbar = 0 1
S R = 0 0 -> Q, Qbar = 0 1
S R = 1 1 -> Q, Qbar = 0 0
bang J-K = phuong trinh dac trung (8 to hop): True


**Reading the result.** Q = 1 (set), hold 1, Q = 0 (reset), hold 0, then (1,1) makes both outputs 0: the forbidden state. The J-K table equals the characteristic equation in all 8 combinations.

## 2. The ripple counter
The 3-bit count sequence, and the total delay and maximum frequency of a 5-bit counter with 15 ns stages.

In [2]:
# Bo dem gon song n bit: moi FF lat khi FF truoc di tu 1 xuong 0
def ripple(n, pulses, start=0):
    q = [(start >> i) & 1 for i in range(n)]
    for _ in range(pulses):
        i = 0
        while i < n:
            q[i] ^= 1
            if q[i] == 1: break
            i += 1
    return sum(b << i for i, b in enumerate(q))
print([format(ripple(3, k), '03b') for k in range(9)])
n, tpd = 5, 15e-9
print('tre tong', n * tpd * 1e9, 'ns; f_max = ', round(1 / (n * tpd) / 1e6, 2), 'MHz; modulus', 2 ** n)
print('Floyd Self-Test 10: 4 FF x 12 ns =', 4 * 12, 'ns')

['000', '001', '010', '011', '100', '101', '110', '111', '000']
tre tong 75.0 ns; f_max =  13.33 MHz; modulus 32
Floyd Self-Test 10: 4 FF x 12 ns = 48 ns


**Reading the result.** The sequence runs 000 to 111 then back to 000. The 5-bit counter: delay 75 ns, f_max ≈ 13.33 MHz, modulus 32. Floyd Self-Test 10: 4 × 12 = 48 ns.

## 3. Shift register, ring and Johnson
The contents after two pulses (Floyd Self-Test 4), the two state sequences, and the serial entry time.

In [3]:
# Thanh ghi dich phai, bit vao o dau trai; bo dem vong va Johnson
def shift_right(reg, bits):                       # bit ngoai cung ben phai cua `bits` vao truoc
    for b in reversed(bits): reg = b + reg[:-1]
    return reg
print('Floyd Self-Test 4 (hai bit ngoai cung ben phai cua 10110101 la 0 va 1):', shift_right('11100100', '01'))
def johnson(n, k):
    r = [0] * n
    for _ in range(k): r = [1 - r[-1]] + r[:-1]
    return ''.join(map(str, r))
def ring(n, k):
    r = [1] + [0] * (n - 1)
    for _ in range(k): r = [r[-1]] + r[:-1]
    return ''.join(map(str, r))
print('Johnson 4:', [johnson(4, k) for k in range(9)]); print('Vong 4   :', [ring(4, k) for k in range(5)])
print('8 bit o 100 kHz:', 8 / 100e3 * 1e6, 'us; tre 24 us can f =', round(8 / 24e-6 / 1e3, 1), 'kHz')

Floyd Self-Test 4 (hai bit ngoai cung ben phai cua 10110101 la 0 va 1): 01111001
Johnson 4: ['0000', '1000', '1100', '1110', '1111', '0111', '0011', '0001', '0000']
Vong 4   : ['1000', '0100', '0010', '0001', '1000']
8 bit o 100 kHz: 80.0 us; tre 24 us can f = 333.3 kHz


**Reading the result.** After two pulses the register holds 01111001. A 4-FF Johnson counter has 8 states, a 4-FF ring counter 4. Entering 8 bits at 100 kHz takes 80 μs; a 24 μs delay needs 333.3 kHz.

## 4. The 74121 and the 555
The capacitor for a 10 μs pulse, the 555 monostable pulse, and the frequency and duty cycle of a 555 astable.

In [4]:
# 74121 va 555
import math
tw = 10e-6; C = tw / (0.7 * 2000); print('74121, R = 2 kohm, tW = 10 us -> C =', round(C * 1e12), 'pF')
print('555 mot nhip 2,2 kohm, 0,01 uF:', round(1.1 * 2200 * 1e-8 * 1e6, 1), 'us')
R1, R2, C = 1e3, 2e3, 0.1e-6
tH, tL = 0.7 * (R1 + R2) * C, 0.7 * R2 * C; T = tH + tL
print('555 dao dong: tH, tL, T =', round(tH * 1e3, 3), round(tL * 1e3, 3), round(T * 1e3, 3), 'ms; f =', round(1 / T), 'Hz (cong thuc 1,44:', round(1.44 / ((R1 + 2 * R2) * C)), 'Hz); duty =', round(tH / T * 100, 1), '%')

74121, R = 2 kohm, tW = 10 us -> C = 7143 pF
555 mot nhip 2,2 kohm, 0,01 uF: 24.2 us
555 dao dong: tH, tL, T = 0.21 0.14 0.35 ms; f = 2857 Hz (cong thuc 1,44: 2880 Hz); duty = 60.0 %


**Reading the result.** C ≈ 7143 pF (matches Floyd 7-11); the 555 monostable gives 24.2 μs (Floyd's example); the astable has T = 0.35 ms, f ≈ 2857 Hz (the 1.44 formula gives 2880 Hz), duty 60 %.

## 5. Modulus, cascading, LUT
Flip-flops for a few moduli, the frequency after a 5, 8, 10, 10 chain, and the function count of a 4-input LUT.

In [5]:
# Bo dem cat cut va mac tang
import math
for N in (6, 10, 12, 13, 60): print('modulus', N, '-> so FF =', math.ceil(math.log2(N)))
mods = [5, 8, 10, 10]; print('10 MHz /', math.prod(mods), '=', 10e6 / math.prod(mods) / 1e3, 'kHz (Floyd Self-Test 14)')
print('32768 Hz qua 15 tang chia doi =', 32768 / 2 ** 15, 'Hz')
print('LUT 4 ngo vao: bit =', 2 ** 4, ', so ham =', 2 ** 2 ** 4)

modulus 6 -> so FF = 3
modulus 10 -> so FF = 4
modulus 12 -> so FF = 4
modulus 13 -> so FF = 4
modulus 60 -> so FF = 6
10 MHz / 4000 = 2.5 kHz (Floyd Self-Test 14)
32768 Hz qua 15 tang chia doi = 1.0 Hz
LUT 4 ngo vao: bit = 16 , so ham = 65536


**Reading the result.** Moduli 6, 10, 12, 13 and 60 need 3, 4, 4, 4 and 6 flip-flops. 10 MHz divided by 4000 gives 2.5 kHz. A 4-input LUT stores 16 bits and implements 65,536 functions.